### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


**Membuat SparkSession**

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas6") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 13:30:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 13:30:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [6]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Jumlah baris df_transaksi: {df_transaksi.count()}")
df_transaksi.printSchema()
print(f"Jumlah baris df_produk: {df_produk.count()}")
df_produk.printSchema()
print(f"Jumlah baris df_ulasan: {df_ulasan.count()}")
df_ulasan.printSchema()

Jumlah baris df_transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris df_produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris df_ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

In [7]:
df_gabung  = df_transaksi.join(df_produk, on="product_id", how="inner").join(df_ulasan, on="order_id", how="left")

df_gabung = df_gabung.withColumn("total_pendapatan",col("unit_terjual") * col("harga"))

print(f"Jumlah baris setelah digabungkan: {df_gabung.count()}")
df_gabung.show(10, truncate=False)

Jumlah baris setelah digabungkan: 5000
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|tanggal            |harga |kategori    |nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|TX0     |21        |2           |2026-10-10 00:00:00|25000 |Elektronik  |Produk-21  |1     |50000           |
|TX1     |8         |6           |2026-10-25 00:00:00|250000|Fashion     |Produk-8   |NULL  |1500000         |
|TX2     |25        |4           |2026-10-13 00:00:00|25000 |Elektronik  |Produk-25  |2     |100000          |
|TX3     |3         |4           |2026-10-18 00:00:00|75000 |Fashion     |Produk-3   |5     |300000          |
|TX4     |19        |6           |2026-10-07 00:00:00|75000 |Rumah Tangga|Produk-19  |NULL  |450000          |
|TX5     |10        |3           |2026-10-01 00:00:00|75000 |Elektronik  

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

In [8]:
df_gabung = df_gabung.withColumn("ada_ulasan", when(col("rating").isNull(), False).otherwise(True))
df_gabung = df_gabung.na.fill({"rating" : 0})

df_gabung.show(10)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
|     TX5|        10|   

kita tahu bahwah rating yang diberikan pelanggan berada pada range 1 hingga 5 sehingga ketika kita memberikan nilai 0 pada rating kita tahu bahwa pelanggan tersebut tidak memberikan rating

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.


In [9]:
df_gabung.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/asta/tugas6/hasil_etl"
)

!hdfs dfs -ls -R /user/asta/tugas6/hasil_etl

df_ulang = spark.read.parquet("hdfs://localhost:9000/user/asta/tugas6/hasil_etl")
df_ulang.count()

-rw-r--r--   3 asta supergroup          0 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - asta supergroup          0 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 asta supergroup      16361 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Elektronik/part-00000-40cf59c0-3367-4c7d-a4b2-70d4ec5aee02.c000.snappy.parquet
drwxr-xr-x   - asta supergroup          0 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 asta supergroup      11131 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Fashion/part-00000-40cf59c0-3367-4c7d-a4b2-70d4ec5aee02.c000.snappy.parquet
drwxr-xr-x   - asta supergroup          0 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 asta supergroup      10389 2026-09-30 13:31 /user/asta/tugas6/hasil_etl/kategori=Kesehatan/part-00000-40cf59c0-3367-4c7d-a4b2-70d4ec5aee02.c000.snappy.parquet
drwxr-xr-x   - asta supergroup          0 2026-09-30 13:31 /user/a

5000

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [10]:
ringkasan_ulasan = df_gabung.groupBy("kategori").agg(
    spark_sum(col("ada_ulasan").cast("int")).alias("total_ulasan"),
    count("*").alias("total_transaksi"))
ringkasan_presentase = ringkasan_ulasan.withColumn(
    "Presentase", col("total_ulasan") / col("total_transaksi") * 100
    ).orderBy("Presentase")
ringkasan_ulasan.show()
ringkasan_presentase.show()

+------------+------------+---------------+
|    kategori|total_ulasan|total_transaksi|
+------------+------------+---------------+
|  Elektronik|        1168|           1653|
|     Fashion|         726|           1035|
|     Makanan|         369|            536|
|Rumah Tangga|         575|            815|
|   Kesehatan|         662|            961|
+------------+------------+---------------+

+------------+------------+---------------+-----------------+
|    kategori|total_ulasan|total_transaksi|       Presentase|
+------------+------------+---------------+-----------------+
|     Makanan|         369|            536|68.84328358208955|
|   Kesehatan|         662|            961|68.88657648283039|
|     Fashion|         726|           1035|70.14492753623188|
|Rumah Tangga|         575|            815| 70.5521472392638|
|  Elektronik|        1168|           1653|  70.659407138536|
+------------+------------+---------------+-----------------+



Hal ini mungkin penting bagi tim marketing karena dapat berpengaruh pada peforma penjualan ke depannya, dari presentase ulasan ini tim marketing bisa mengevalusai kategori apa saja yang perlu ditingkatkan kedepannya. Ulasan ini sangatlah penting terutama jika kita melakukan penjualannya melalui e-commerce, bagi kebanyakan pelanggan ulasan dari suatu produk menunjukkan kualitas dari produk yang dijual, jika pada hasil presentase ulasan suatu produk rendah ada kemungkinan pelanggan lain berasumsi bahwa produk yang dijual kualitasnya buruk sehingga ini dapat menjadi bahan evaluasi bagi tim marketing. Untuk meningkatkan ulasan terutama pada produk makanan kita dapat memberikan diskon bagi pelanggan yang memberikan ulasan, hal ini dapat meningkatkan presentase ulasan dan kepercayaan pelanggan pada produk kita.

**Eksplorasi**

Cek apakah order_id ada duplikasi

In [14]:
df_cek_duplikat = df_transaksi.groupBy("order_id").agg(count("order_id").alias("jumlah")).filter(col("jumlah") > 1)

print(f"Jumlah order_id duplikat: {df_cek_duplikat.count()}")
df_cek_duplikat.show()

Jumlah order_id duplikat: 0
+--------+------+
|order_id|jumlah|
+--------+------+
+--------+------+



In [22]:
print(f"Jumlah baris transaksi: {df_transaksi.count()}")
print(f"jumlah baris order_id: {df_transaksi.select('order_id').distinct().count()}")

Jumlah baris transaksi: 5000
jumlah baris order_id: 5000


**Menutup SparkSession**

In [23]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
